<a href="https://colab.research.google.com/github/tusharastirmind-commits/Info5731_Python/blob/main/In_class_exercises_4_Text_Cleaning_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Data Preprocessing & Cleaning (Text)  
**Time:** 20 minutes  |  **Points:** 10  

## Instructions
- This is an individual in-class assignment.  
- Write your code **inside each answer cell**.  
- Print the required outputs.  
- Submit your GitHub/Colab link as instructed by the instructor.


You are given a small dataset of customer support messages as a **TAB-separated text file**:  
- `support_messages.txt`

You will download this file from **Canvas** and upload it to your **Google Colab** notebook.

**How to upload it to your Google Colab notebook?**

1. Download `support_messages.txt` from Canvas.
3. In **the left sidebar**, click the **Files** icon (folder).  
4. Click **Upload** and select `support_messages.txt`.

6. RightAfter uploading, the file will appear in the Colab file list on the left.

6. Right-click the file, copy its path, and paste it into the FILE_PATH variable in Q1.

7. Run Q1 to load the dataset.



> Important: Keep the file name exactly as `support_messages.txt`.


## Questions (Total = 10 points)

### Q1 (1 point) — Load the dataset
Load the TAB-separated file into a pandas DataFrame with columns: `id`, `message`.  
Print: **(a)** `df.shape`, **(b)** `df.head(3)`.

### Q2 (3 points) — Descriptive columns
Add these columns for each message and print the full DataFrame:
- `word_count`: number of words  
- `char_count`: number of characters  
- `num_count`: number of digits (0–9)  
- `upper_word_count`: number of ALL-CAPS words (e.g., `"WHY"`, `"DAMAGED"`)  

### Q3 (3 points) — Clean text
Build a `clean_text(text)` function and create a new column `clean` with these steps **in order**:
1) lowercase  
2) remove punctuation/symbols (keep letters/numbers/spaces)  
3) remove English stopwords (use **nltk** or **sklearn** list)  
4) remove extra spaces  

Print the **original** message and **clean** version for rows `id=1` and `id=4`.

### Q4 (2 points) — Regex extraction
Using RegEx, extract and create two new columns:
- `order_id`: first occurrence of pattern `ORD-####` (case-insensitive; `ord-1060` is valid)  
- `email`: first email address if present (otherwise `None`/`NaN`)  

Print: `id`, `order_id`, `email` for all rows.

### Q5 (1 point) — TF-IDF keywords
Using the `clean` column, compute **TF-IDF** for the messages and print the **top 5 keywords** with the highest **average TF-IDF** across documents.


In [3]:
import re
import pandas as pd
import numpy as np
import nltk
from google.colab import files

## Q1 (1 point) — Answer below

In [4]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name, sep="\t")

print(df.shape)
print(df.head(3))

Saving support_messages.txt to support_messages.txt
(8, 2)
   id                                            message
0   1  Hi!! My ORDER is late :(  Order# ORD-1042. Ema...
1   2  Refund please!!! I was charged 2 times... invo...
2   3        Great service, thanks! arrived in 2 days :)


## Q2 (3 points) — Answer below

In [5]:
# Q2 — ANSWER CELL
df["word_count"] = df["message"].apply(lambda x: len(str(x).split()))
df["char_count"] = df["message"].apply(lambda x: len(str(x)))
df["num_count"] = df["message"].str.count(r"\d")
df["upper_word_count"] = df["message"].apply(lambda x: sum(1 for w in str(x).split() if w.isupper()))

print(df[["id", "message", "word_count", "char_count", "num_count", "upper_word_count"]].to_string(index=False))

 id                                                                      message  word_count  char_count  num_count  upper_word_count
  1    Hi!! My ORDER is late :(  Order# ORD-1042. Email me at sara.Ali@gmail.com          12          73          4                 2
  2      Refund please!!! I was charged 2 times... invoice ORD-1042 and ORD-1043          11          71          9                 3
  3                                  Great service, thanks! arrived in 2 days :)           8          43          1                 0
  4                      WHY is my package DAMAGED??? tracking says delivered...           8          55          0                 2
  5          Need to change address: 7421 Frankford Rd Apt 1232, Dallas TX 75252          12          67         13                 1
  6                   Support ticket: ORD-1050. Call me at (469) 555-0182 ASAP!!           9          58         14                 2
  7 I can’t login— password reset link not working. email: meh

## Q3 (3 points) — Answer below

In [6]:
# Q3 — ANSWER CELL
# Option A (sklearn stopwords):
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# STOPWORDS = set(ENGLISH_STOP_WORDS)

# Option B (nltk stopwords):
# import nltk
# nltk.download("stopwords")
# from nltk.corpus import stopwords
# STOPWORDS = set(stopwords.words("english"))

# Q3 — ANSWER CELL
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    # Step 1: lowercase
    text = text.lower()
    # Step 2: remove punctuation/symbols (keep letters, numbers, spaces)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    # Step 3: remove stopwords
    text = " ".join(w for w in text.split() if w not in STOPWORDS)
    # Step 4: remove extra spaces
    text = re.sub(r" +", " ", text).strip()
    return text

df["clean"] = df["message"].apply(clean_text)

# Print original and clean for id=1 and id=4
for row_id in [1, 4]:
    row = df[df["id"] == row_id].iloc[0]
    print(f"--- id={row_id} ---")
    print("Original:", row["message"])
    print("Clean:   ", row["clean"])
    print()




--- id=1 ---
Original: Hi!! My ORDER is late :(  Order# ORD-1042. Email me at sara.Ali@gmail.com
Clean:    hi order late order ord1042 email saraaligmailcom

--- id=4 ---
Original: WHY is my package DAMAGED??? tracking says delivered...
Clean:    package damaged tracking says delivered



## Q4 (2 points) — Answer below

In [7]:
# Q4 — ANSWER CELL
# order_id pattern: r"ORD-\d{4}" with re.IGNORECASE
# email pattern (simple): r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# TODO: create df["order_id"] and df["email"]
# TODO: print/display df[["id", "order_id", "email"]]

# 4) Cleaning pipeline (answers: "what preprocessing steps should be applied?")
# -----------------------------
# Q4 — ANSWER CELL
def extract_order_id(text):
    match = re.search(r"ORD-\d{4}", text, re.IGNORECASE)
    return match.group(0) if match else None

def extract_email(text):
    match = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)
    return match.group(0) if match else None

df["order_id"] = df["message"].apply(extract_order_id)
df["email"] = df["message"].apply(extract_email)

print(df[["id", "order_id", "email"]].to_string(index=False))






 id order_id                 email
  1 ORD-1042    sara.Ali@gmail.com
  2 ORD-1042                  None
  3     None                  None
  4     None                  None
  5     None                  None
  6 ORD-1050                  None
  7     None mehri.sattari@unt.edu
  8 ord-1060                  None


## Q5 (1 point) — Answer below

In [8]:
# -----------------------------
# 5) Save cleaned output
# -----------------------------

# Q5 — ANSWER CELL
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["clean"])

avg_tfidf = np.mean(tfidf_matrix.toarray(), axis=0)
feature_names = vectorizer.get_feature_names_out()

top5_idx = np.argsort(avg_tfidf)[::-1][:5]
print("Top 5 TF-IDF keywords:")
for idx in top5_idx:
    print(f"  {feature_names[idx]}: {avg_tfidf[idx]:.4f}")

Top 5 TF-IDF keywords:
  order: 0.1135
  ord1042: 0.0795
  email: 0.0768
  says: 0.0559
  package: 0.0559


## Grading Checklist
- Q1: correct load + prints  
- Q2: correct counts  
- Q3: cleaning follows the required order + prints for id=1 and id=4  
- Q4: regex extraction works (case-insensitive `ORD-####` and emails)  
- Q5: prints 5 keywords + their scores (rounding is fine)
